# Pipeline Entrega 23-F

Notebook unico para ejecutar todo el flujo end-to-end:
1. Scraping de La Moncloa
2. Descarga de PDFs
3. Scraping del catalogo RTVE
4. Extraccion de texto + OCR + corpus base (pipeline.py)
5. Construccion de corpus analitico con calidad OCR (build_corpus.py)
6. Entregables Sprint 1 de Alber (muestra manual + vocabulario NLP)

Este notebook fusiona el flujo operativo de main.py y pipeline.py en ejecucion por celdas.

Este notebook genera:

- data/metadata/documents_enriched.csv
- data/metadata/document_corpus.csv
- outputs/sprint1/manual_validation_sample.csv
- outputs/sprint1/top_words_overall.csv
- outputs/sprint1/top_words_by_ministry.csv
- outputs/sprint1/nlp_preprocess_summary.txt

In [1]:
from pathlib import Path
import os
import sys

# Detectar raiz del repo de forma robusta, ejecutes desde donde ejecutes.
CANDIDATES = [Path.cwd(), *Path.cwd().parents]
ROOT = None
for p in CANDIDATES:
    if (p / 'main.py').exists() and (p / 'pipeline.py').exists():
        ROOT = p
        break

if ROOT is None:
    raise RuntimeError('No se encontro la raiz del proyecto (main.py/pipeline.py).')

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src' / 'data_etl'))

print(f'ROOT: {ROOT}')
print(f'Python: {sys.executable}')

ROOT: c:\Users\Alber\project-23f
Python: c:\Users\Alber\project-23f\.venv\Scripts\python.exe


In [ ]:
# Imports principales del flujo (main.py + pipeline.py + sprint1).
import pandas as pd

from main import run_scraper, run_download, show_status
from pipeline import run as run_pipeline
from src.data_etl.rtve_scraper import scrape_all
from src.data_etl.build_corpus import build_corpus
from src.sprint1.manual_validation_sample import build_sample

# Parametros operativos (ajustables).
MAX_EXTRACTION_WORKERS = 4
MAX_OCR_WORKERS = 4
MANUAL_SAMPLE_N = 15
MANUAL_SAMPLE_SEED = 23
TOP_K = 30
MIN_LEN = 3

print('Configuracion cargada.')

Configuracion cargada.


## 0) Estado inicial

In [4]:
show_status()


PROJECT STATUS

Downloaded PDFs: 0 (directory does not exist)
Extracted texts: 0 (directory does not exist)
Detected links: 0 (scraping has not been done)


## 1) Scraping La Moncloa (main.py --scrape)

Genera data/metadata/moncloa_links.csv

In [5]:
ok_scrape = run_scraper()
if not ok_scrape:
    raise RuntimeError('Fallo en scraping de La Moncloa.')


STEP 1: LA MONCLOA SCRAPING
Accessing: https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23f.aspx

Total documents found: 155

By Ministry:
 ministry
Defensa       108
Interior       28
Exteriores     19

By Category:
 category
General    155
Saved to: data/metadata/moncloa_links.csv

✓ Scraping completed: 155 PDFs detected


## 2) Descarga PDFs (main.py --download)

Descarga a data/raw/ y registra data/metadata/download_log.csv

In [6]:
ok_download = run_download()
if not ok_download:
    raise RuntimeError('Fallo en descarga de PDFs.')


STEP 2: PDF DOWNLOAD
DOWNLOADER - 23-F Documents La Moncloa

Total PDFs to download : 155
Destination directory  : data\raw
Concurrent threads     : 3
Rate limit per thread  : 0.5s



Downloading: 100%|██████████| 155/155 [00:45<00:00,  3.43it/s]


SUMMARY
status
success    155

✓ 155 PDFs  (284.6 MB)

Log: data\metadata\download_log.csv

✓ Download completed: 155/155 PDFs


## 3) Scraping RTVE

Genera data/metadata/rtve_documents.csv

In [8]:
rtve_df = scrape_all()
if rtve_df.empty:
    raise RuntimeError('No se pudo extraer RTVE o la tabla vino vacia.')

rtve_out = ROOT / 'data' / 'metadata' / 'rtve_documents.csv'
rtve_out.parent.mkdir(parents=True, exist_ok=True)
rtve_df.to_csv(rtve_out, index=False, encoding='utf-8-sig')
print(f'RTVE docs guardados: {len(rtve_df)} en {rtve_out}')

=== Scraper 23F RTVE ===


[Page 1]
  → 167 resultados
  → Página 1 de 1
  → Accumulated: 167

✅ Total documents in listing: 167

📄 Fetching summaries (167 pages, 8 threads)...


Details:  34%|███▍      | 57/167 [00:30<00:54,  2.03it/s]14:41:07  WARNING   Retrying (Retry(total=0, connect=None, read=False, redirect=None, status=None)) after connection broken by 'MustDowngradeError('The server yielded its support for HttpVersion.h3 through the Alt-Svc header while unable to do so. To remediate that issue, either disable HttpVersion.h3 or reach out to the server admin.')': /document/ocr/1859
14:41:07  WARNING   Retrying (Retry(total=0, connect=None, read=False, redirect=None, status=None)) after connection broken by 'MustDowngradeError('The server yielded its support for HttpVersion.h3 through the Alt-Svc header while unable to do so. To remediate that issue, either disable HttpVersion.h3 or reach out to the server admin.')': /document/ocr/1858
14:41:07  WARNING   Retrying (Retry(total=0, connect=None, read=False, redirect=None, status=None)) after connection broken by 'MustDowngradeError('The server yielded its support for HttpVersion.h3 through the Alt-Svc heade

RTVE docs guardados: 167 en c:\Users\Alber\project-23f\data\metadata\rtve_documents.csv


## 4) Pipeline principal (pipeline.py --ocr)

Hace extraccion de texto, OCR en escaneados/fotos y construye un corpus base. Poner apply_ocr a False para solo extraer texto de PDFs no escaneados. Como resultado se obtiene un dataset más pequeño pero es muchísimo más rápido de procesar y analizar, con calidad de texto suficiente para muchos casos y comprobar el correcto funcionamiento del pipeline sin esperar a la parte de OCR, que es la más costosa.

In [9]:
df_final = run_pipeline(
    apply_ocr=True,
    max_extraction_workers=MAX_EXTRACTION_WORKERS,
    max_ocr_workers=MAX_OCR_WORKERS,
    force_reextract=False,
    save=True,
)

print('df_final shape:', df_final.shape)
print('Moncloa docs:', (df_final['source'] == 'Moncloa').sum())
print('RTVE docs:', (df_final['source'] == 'RTVE').sum())

14:41:21  INFO      Loaded: 155 Moncloa links, enriching with extraction results
14:41:21  INFO      Loaded: 155 Moncloa docs, 167 RTVE docs
14:41:21  INFO      === Step 1: PDF text extraction (pdfplumber) ===
14:41:21  INFO        155 PDFs to extract (4 threads)
Extracting: 100%|██████████| 155/155 [00:36<00:00,  4.19it/s]
14:41:58  INFO        Done: 155 extracted, 0 errors
14:42:16  INFO      === Step 2: OCR for scanned PDFs (EasyOCR) ===
14:42:17  WARNING   Using CPU. Note: This module is much faster with a GPU.
14:42:17  WARNING   Downloading detection model, please wait. This may take several minutes depending upon your network connection.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

14:42:20  INFO      Download complete
14:42:20  WARNING   Downloading recognition model, please wait. This may take several minutes depending upon your network connection.


Progress: |█████████████████████████████████████████████████-| 99.9% Complete

14:42:23  INFO      Download complete.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

14:42:26  INFO        OCR backend: cpu
14:42:26  INFO        105 PDFs to OCR (4 processes)
OCR: 100%|██████████| 105/105 [1:29:47<00:00, 51.31s/it]  
16:12:17  INFO        Done: 105 OCR'd, 0 errors
16:12:17  INFO      === Step 3: Loading extracted texts ===
16:12:18  INFO        155/155 texts loaded (0 missing)
16:12:18  INFO      Saved: c:\Users\Alber\project-23f\data\metadata\documents_enriched.csv  (155 rows)
16:12:18  INFO      === Step 4: Building final corpus ===
16:12:18  INFO        Total: 322 documents (Moncloa: 155, RTVE: 167)
16:12:18  INFO      Saved: c:\Users\Alber\project-23f\data\metadata\document_corpus.csv  (322 rows)


df_final shape: (322, 13)
Moncloa docs: 155
RTVE docs: 167


## 5) Corpus analitico enriquecido (build_corpus.py)

Re-escribe document_corpus.csv con columnas de calidad OCR y texto canonico para NLP:
- ocr_quality_score
- flag_illegible
- analysis_text

In [10]:
build_corpus(extract_metadata=False)

df_corpus = pd.read_csv('data/metadata/document_corpus.csv')
print('document_corpus shape:', df_corpus.shape)
print('columns:', df_corpus.columns.tolist())
if 'flag_illegible' in df_corpus.columns:
    print('Moncloa ilegibles:', int(df_corpus[(df_corpus['source'] == 'Moncloa') & (df_corpus['flag_illegible'] == True)].shape[0]))

Loading Moncloa documents...
  155 documents loaded.
Loading RTVE documents...
  167 documents loaded.

Scanning corpus for OCR quality...
Data cleaning completed: identified 8 potentially illegible documents.

Applying simple rules to infer doc_type...

Normalizing dates and calculating precision...

Corpus saved to data/metadata/document_corpus.csv
Total documents : 322
  Moncloa (legible) : len=147
  Moncloa (illegible) : len=8
  RTVE          : 167
document_corpus shape: (322, 17)
columns: ['doc_id', 'source', 'title', 'url', 'ministry', 'category', 'filename', 'date', 'date_precision', 'doc_type', 'tags', 'extracted_text', 'extracted_text_length', 'rtve_summary', 'ocr_quality_score', 'flag_illegible', 'analysis_text']
Moncloa ilegibles: 8


## 6) Sprint 1 Alber - Muestra de validacion manual

Genera outputs/sprint1/manual_validation_sample.csv

In [ ]:
sample = build_sample(sample_size=MANUAL_SAMPLE_N, seed=MANUAL_SAMPLE_SEED)
out_sample = ROOT / 'outputs' / 'sprint1' / 'manual_validation_sample.csv'
out_sample.parent.mkdir(parents=True, exist_ok=True)
sample.to_csv(out_sample, index=False, encoding='utf-8')
print(f'Muestra guardada: {out_sample}')
print(f'Filas: {len(sample)}')
display(sample.head(10))

## 7) Sprint 1 Alber - Vocabulario NLP por ministerio

Ejecutamos el script del sprint para producir:
- outputs/sprint1/top_words_overall.csv
- outputs/sprint1/top_words_by_ministry.csv
- outputs/sprint1/nlp_preprocess_summary.txt

In [11]:
import subprocess

cmd = [
    sys.executable,
    str(ROOT / 'src' / 'sprint1' / 'nlp_vocab_by_ministry.py'),
    '--top-k', str(TOP_K),
    '--min-len', str(MIN_LEN),
]
result = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Fallo en nlp_vocab_by_ministry.py')

Saved:
- C:\Users\Alber\project-23f\outputs\sprint1\top_words_overall.csv
- C:\Users\Alber\project-23f\outputs\sprint1\top_words_by_ministry.csv
- C:\Users\Alber\project-23f\outputs\sprint1\nlp_preprocess_summary.txt



## 8) Verificacion final de entregables

In [ ]:
from pathlib import Path

expected = [
    ROOT / 'data' / 'metadata' / 'moncloa_links.csv',
    ROOT / 'data' / 'metadata' / 'download_log.csv',
    ROOT / 'data' / 'metadata' / 'rtve_documents.csv',
    ROOT / 'data' / 'metadata' / 'documents_enriched.csv',
    ROOT / 'data' / 'metadata' / 'document_corpus.csv',
    ROOT / 'outputs' / 'sprint1' / 'manual_validation_sample.csv',
    ROOT / 'outputs' / 'sprint1' / 'top_words_overall.csv',
    ROOT / 'outputs' / 'sprint1' / 'top_words_by_ministry.csv',
    ROOT / 'outputs' / 'sprint1' / 'nlp_preprocess_summary.txt',
]

for path in expected:
    print(f'[OK]' if path.exists() else '[MISSING]', path)

show_status()

## Opcional - Reprocesar un PDF escaneado concreto con OCR

Si algun PDF escaneado queda con texto vacio o muy pobre, puedes reprocesarlo por filename (ejemplo del Documento_38_R):

```python
import subprocess, sys
subprocess.run([
    sys.executable,
    str(ROOT / 'src' / 'sprint1' / 'reprocess_single_ocr.py'),
    '--filename', 'Documento_38_R.pdf',
    '--resolution', '100'
], check=True)
```

Luego vuelve a ejecutar la celda 5 (build_corpus) para refrescar el corpus final.